# LAB | Imbalanced

**Load the data**

In this challenge, we will be working with Credit Card Fraud dataset.

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv

Metadata

- **distance_from_home:** the distance from home where the transaction happened.
- **distance_from_last_transaction:** the distance from last transaction happened.
- **ratio_to_median_purchase_price:** Ratio of purchased price transaction to median purchase price.
- **repeat_retailer:** Is the transaction happened from same retailer.
- **used_chip:** Is the transaction through chip (credit card).
- **used_pin_number:** Is the transaction happened by using PIN number.
- **online_order:** Is the transaction an online order.
- **fraud:** Is the transaction fraudulent. **0=legit** -  **1=fraud**


In [2]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

# Import imbalanced-learn components for resampling
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler


In [2]:
fraud = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv")
fraud.head()

,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price,repeat_retailer,used_chip,used_pin_number,online_order,fraud
0,57.877857,0.311140,1.945940,1.0,1.0,0.0,0.0,0.0
1,10.829943,0.175592,1.294219,1.0,0.0,0.0,0.0,0.0
2,5.091079,0.805153,0.427715,1.0,0.0,0.0,1.0,0.0
3,2.247564,5.600044,0.362663,1.0,1.0,0.0,1.0,0.0
4,44.190936,0.566486,2.222767,1.0,1.0,0.0,1.0,0.0


**Steps:**

- **1.** What is the distribution of our target variable? Can we say we're dealing with an imbalanced dataset?
- **2.** Train a LogisticRegression.
- **3.** Evaluate your model. Take in consideration class importance, and evaluate it by selection the correct metric.
- **4.** Run **Oversample** in order to balance our target variable and repeat the steps above, now with balanced data. Does it improve the performance of our model? 
- **5.** Now, run **Undersample** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model?
- **6.** Finally, run **SMOTE** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model? 

In [4]:
# ### Credit Card Fraud Detection: Resampling & Evaluation Pipeline


# Display shape and head
print(f"Dataset Shape: {fraud.shape}")
print(fraud.head())

# Check distribution of the target variable
target_counts = fraud['fraud'].value_counts()
target_pct = fraud['fraud'].value_counts(normalize=True) * 100

print("\n--- Target Variable Distribution ---")
for idx in target_counts.index:
    print(f"Class {idx}: {target_counts[idx]} counts ({target_pct[idx]:.2f}%)")

print("\nConclusion: The dataset is highly imbalanced (~8.7% fraud vs ~91.3% legitimate).")


Dataset Shape: (1000000, 8)
   distance_from_home  distance_from_last_transaction  \
0           57.877857                        0.311140   
1           10.829943                        0.175592   
2            5.091079                        0.805153   
3            2.247564                        5.600044   
4           44.190936                        0.566486   

   ratio_to_median_purchase_price  repeat_retailer  used_chip  \
0                        1.945940              1.0        1.0   
1                        1.294219              1.0        0.0   
2                        0.427715              1.0        0.0   
3                        0.362663              1.0        1.0   
4                        2.222767              1.0        1.0   

   used_pin_number  online_order  fraud  
0              0.0           0.0    0.0  
1              0.0           0.0    0.0  
2              0.0           1.0    0.0  
3              0.0           1.0    0.0  
4              0.0          

In [5]:
# Separate features and target
X = fraud.drop(columns=['fraud'])
y = fraud['fraud']

# Split into train and test sets (stratify ensures class distribution is maintained)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale features (essential for stable Logistic Regression convergence)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [7]:
 # Helper Function to Train and Evaluate Models


def evaluate_model(X_tr, y_tr, X_te, y_te, model_name="Model"):
    """
    Fits a Logistic Regression model on the training data, 
    predicts on the test set, and prints the classification evaluation.
    """
    print(f"\n==================== {model_name} ====================")
    
    # Train Logistic Regression
    lr = LogisticRegression(max_iter=1000, random_state=42)
    lr.fit(X_tr, y_tr)
    
    # Predict
    y_pred = lr.predict(X_te)
    
    # Print metrics focused on Fraud class (1)
    print("Confusion Matrix:")
    print(confusion_matrix(y_te, y_pred))
    
    print("\nClassification Report:")
    print(classification_report(y_te, y_pred, target_names=['Legitimate (0)', 'Fraud (1)']))

In [8]:
# #### Steps 2 & 3: Train & Evaluate Baseline Logistic Regression (Imbalanced Data)


evaluate_model(X_train_scaled, y_train, X_test_scaled, y_test, model_name="Baseline Model (Imbalanced)")


==================== Baseline Model (Imbalanced) ====================
Confusion Matrix:
[[181296   1223]
 [  6895  10586]]

Classification Report:
                precision    recall  f1-score   support

Legitimate (0)       0.96      0.99      0.98    182519
     Fraud (1)       0.90      0.61      0.72     17481

      accuracy                           0.96    200000
     macro avg       0.93      0.80      0.85    200000
  weighted avg       0.96      0.96      0.96    200000



In [9]:
# #### Step 4: Random Oversampling (ROS)


# Run oversampling on training data
ros = RandomOverSampler(random_state=42)
X_train_ros, y_train_ros = ros.fit_resample(X_train_scaled, y_train)

print(f"Original train shape: {X_train_scaled.shape}, Resampled train shape: {X_train_ros.shape}")
print(f"Resampled target value counts:\n{y_train_ros.value_counts()}")

# Evaluate model with oversampled data
evaluate_model(X_train_ros, y_train_ros, X_test_scaled, y_test, model_name="Logistic Regression + Random Oversampling")

Original train shape: (800000, 7), Resampled train shape: (1460156, 7)
Resampled target value counts:
fraud
0.0    730078
1.0    730078
Name: count, dtype: int64

==================== Logistic Regression + Random Oversampling ====================
Confusion Matrix:
[[170390  12129]
 [   911  16570]]

Classification Report:
                precision    recall  f1-score   support

Legitimate (0)       0.99      0.93      0.96    182519
     Fraud (1)       0.58      0.95      0.72     17481

      accuracy                           0.93    200000
     macro avg       0.79      0.94      0.84    200000
  weighted avg       0.96      0.93      0.94    200000



In [10]:
# #### Step 5: Random Undersampling (RUS)


# Run undersampling on training data
rus = RandomUnderSampler(random_state=42)
X_train_rus, y_train_rus = rus.fit_resample(X_train_scaled, y_train)

print(f"Original train shape: {X_train_scaled.shape}, Resampled train shape: {X_train_rus.shape}")
print(f"Resampled target value counts:\n{y_train_rus.value_counts()}")

# Evaluate model with undersampled data
evaluate_model(X_train_rus, y_train_rus, X_test_scaled, y_test, model_name="Logistic Regression + Random Undersampling")


Original train shape: (800000, 7), Resampled train shape: (139844, 7)
Resampled target value counts:
fraud
0.0    69922
1.0    69922
Name: count, dtype: int64

==================== Logistic Regression + Random Undersampling ====================
Confusion Matrix:
[[170394  12125]
 [   918  16563]]

Classification Report:
                precision    recall  f1-score   support

Legitimate (0)       0.99      0.93      0.96    182519
     Fraud (1)       0.58      0.95      0.72     17481

      accuracy                           0.93    200000
     macro avg       0.79      0.94      0.84    200000
  weighted avg       0.96      0.93      0.94    200000



In [11]:
# Step 6: SMOTE (Synthetic Minority Over-sampling Technique)

# %%
# Run SMOTE on training data
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print(f"Original train shape: {X_train_scaled.shape}, Resampled train shape: {X_train_smote.shape}")
print(f"Resampled target value counts:\n{y_train_smote.value_counts()}")


Original train shape: (800000, 7), Resampled train shape: (1460156, 7)
Resampled target value counts:
fraud
0.0    730078
1.0    730078
Name: count, dtype: int64


In [12]:
# Evaluate model with SMOTE data
evaluate_model(X_train_smote, y_train_smote, X_test_scaled, y_test, model_name="Logistic Regression + SMOTE")


==================== Logistic Regression + SMOTE ====================
Confusion Matrix:
[[170386  12133]
 [   907  16574]]

Classification Report:
                precision    recall  f1-score   support

Legitimate (0)       0.99      0.93      0.96    182519
     Fraud (1)       0.58      0.95      0.72     17481

      accuracy                           0.93    200000
     macro avg       0.79      0.94      0.84    200000
  weighted avg       0.96      0.93      0.94    200000

